# RQ1 Annotation Reliability Walkthrough

This notebook reproduces Section 4.1 of the paper: human--LLM agreement, outcome-conditioned agreement, and the empirically grounded annotation policy.

The calculations use the 30% human-consensus gold subset and the combined LLM-v1 annotations. Only cases that appear in the human gold subset are used for agreement metrics.

In [1]:
from pathlib import Path
import sys
import pandas as pd

# Resolve repository root even when Jupyter starts in a different working directory.
ROOT = Path.cwd()
for candidate in [ROOT, *ROOT.parents]:
    if (candidate / "RQ2_Prompt_Effectiveness_Modeling").exists() and (candidate / "Dataset_Construction").exists():
        ROOT = candidate
        break
sys.path.insert(0, str(ROOT))
print(ROOT)

/Users/richardsserunjogi/Work/UNLV/PatchPrompt


## 1. Load and align validation data

The human file contains the stratified 30% gold-standard subset. The LLM-v1 file contains all annotated cases, so the script aligns it to the human subset by `Case_ID`.

In [2]:
from RQ1_Prompt_Evaluation_Validation.analysis.rq1.agreement_analysis import load_validation_data

validation_df = load_validation_data(ROOT)
validation_df[['Case_ID','Classification','Human_Context','LLM_Context','Human_Specificity','LLM_Specificity','Human_Verification','LLM_Verification']].head()

,Case_ID,Classification,Human_Context,LLM_Context,Human_Specificity,LLM_Specificity,Human_Verification,LLM_Verification
0,CL-10,CL,2,2.0,1,1.0,1,1.0
1,CL-15,CL,2,2.0,1,1.0,0,1.0
2,CL-24,CL,2,2.0,1,1.0,0,1.0
3,CL-29,CL,2,2.0,1,2.0,1,1.0
4,CL-30,CL,2,2.0,1,1.0,1,1.0


In [3]:
validation_df['Classification'].value_counts().reindex(['PA','PN','CL','NE'])

Classification
PA    27
PN    16
CL    13
NE    24
Name: count, dtype: int64

## 2. Overall human--LLM agreement

This reproduces RQ1a using quadratic weighted Cohen's κ, mean absolute error, and directional bias. Directional bias is `LLM - Human`, so negative values indicate LLM under-scoring.

In [4]:
from RQ1_Prompt_Evaluation_Validation.analysis.rq1.agreement_analysis import compute_overall_agreement

overall = compute_overall_agreement(validation_df)
overall

,Dimension,Quadratic_Weighted_Kappa,MAE,Directional_Bias
0,Context,0.526,0.475,-0.450
1,Specificity,0.431,0.325,-0.200
2,Verification,0.316,0.338,-0.037


## 3. Outcome-conditioned agreement

This reproduces the per-class κ values reported in Table 1 of the paper.

In [5]:
from RQ1_Prompt_Evaluation_Validation.analysis.rq1.agreement_analysis import compute_class_conditioned_kappa

class_kappa = compute_class_conditioned_kappa(validation_df)
class_kappa

,Class,N,Quadratic_Weighted_Kappa_C,Quadratic_Weighted_Kappa_S,Quadratic_Weighted_Kappa_V
0,PA,27,0.427,0.053,0.073
1,PN,16,0.308,0.282,0.667
2,CL,13,0.544,0.581,-0.020
3,NE,24,0.621,0.438,-0.075


## 4. Annotation policy

This reproduces Table 2. The policy is intentionally class-aware and encodes the paper's empirical judgment: Context remains human-scored because of systematic under-scoring; Specificity is automated outside PA; Verification is automated only for PN.

In [6]:
from RQ1_Prompt_Evaluation_Validation.analysis.rq1.agreement_analysis import derive_policy_table

policy = derive_policy_table()
policy

,Metric,PA,PN,CL,NE
0,Context,Human,Human,Human,Human
1,Specificity,Human,Human,LLM,LLM
2,Verification,Human,LLM,Human,Human


## 5. Regenerate RQ1 outputs

The same function is called by `make reproduce`. It writes CSV outputs under `RQ1_Prompt_Evaluation_Validation/results/rq1/`.

In [7]:
from RQ1_Prompt_Evaluation_Validation.analysis.rq1.agreement_analysis import run

outputs = run(ROOT)
list(outputs.keys())

['human_human_records',
 'human_human_disagreements',
 'human_human_metrics',
 'overall',
 'class_kappa',
 'policy',
 'validation_pairs']

## 6. Human--human inter-rater agreement (independent 30% subset)

This section reproduces the human--human reliability analysis using the independent annotation files (before reconciliation/adjudication).

It computes:

- prompt-level agreement records by class and rubric dimension,
- a disagreements-only view,
- category-by-dimension agreement metrics (percent agreement, Cohen's $\kappa$, quadratic weighted $\kappa$).

In [8]:
from RQ1_Prompt_Evaluation_Validation.analysis.rq1.agreement_analysis import (
    load_human_human_annotations,
    compute_human_human_agreement_records,
    compute_human_human_agreement_metrics,
)

human_human_df = load_human_human_annotations(ROOT)
human_human_records = compute_human_human_agreement_records(human_human_df)
human_human_metrics = compute_human_human_agreement_metrics(human_human_records)

human_human_metrics

,Category,Dimension,N,Agreements,Disagreements,Percent_Agreement,Cohens_Kappa,Quadratic_Weighted_Kappa,Measured_Before_Discussion
0,PA,Context,27,16,11,59.3,0.233,0.125,True
1,PA,Specificity,27,19,8,70.4,0.368,0.195,True
2,PA,Verification,27,18,9,66.7,0.399,0.467,True
3,PN,Context,16,9,7,56.2,0.337,0.416,True
4,PN,Specificity,16,6,10,37.5,-0.006,0.279,True
5,PN,Verification,16,12,4,75.0,0.158,0.429,True
6,CL,Context,13,5,8,38.5,0.010,-0.059,True
7,CL,Specificity,13,3,10,23.1,-0.262,-0.573,True
8,CL,Verification,13,4,9,30.8,0.086,0.114,True
9,NE,Context,24,13,11,54.2,0.298,0.563,True


In [9]:
human_human_disagreements = human_human_records[~human_human_records["Agreement"]].copy()

print(f"Total human-human comparisons: {len(human_human_records)}")
print(f"Disagreements: {len(human_human_disagreements)}")

human_human_disagreements.head(10)

Total human-human comparisons: 240
Disagreements: 96


,Category,Case_ID,PR_Link,Conversation_Link,Dimension,Annotator_1,Annotator_2,Annotator_1_Score,Annotator_2_Score,Agreement,Score_Difference,Absolute_Difference,Measured_Before_Discussion
2,PA,PA-33,https://github.com/uchicago-cs/chigame/pull/265,https://chat.openai.com/share/e3e3871f-4acd-49...,Verification,Richard,Daniel,1,2,False,-1,1,True
6,PA,PA-24,https://github.com/viets-software-club/truffle...,https://chat.openai.com/share/48bd44b4-13a7-4f...,Context,Richard,Daniel,1,2,False,-1,1,True
8,PA,PA-24,https://github.com/viets-software-club/truffle...,https://chat.openai.com/share/48bd44b4-13a7-4f...,Verification,Richard,Daniel,1,0,False,1,1,True
9,PA,PA-34,https://github.com/viets-software-club/truffle...,https://chat.openai.com/share/48bd44b4-13a7-4f...,Context,Richard,Daniel,1,2,False,-1,1,True
11,PA,PA-34,https://github.com/viets-software-club/truffle...,https://chat.openai.com/share/48bd44b4-13a7-4f...,Verification,Richard,Daniel,1,0,False,1,1,True
13,PA,PA-20,https://github.com/firezone/firezone/pull/3621,https://chat.openai.com/share/b1bb1f8c-b376-42...,Specificity,Richard,Daniel,1,2,False,-1,1,True
14,PA,PA-20,https://github.com/firezone/firezone/pull/3621,https://chat.openai.com/share/b1bb1f8c-b376-42...,Verification,Richard,Daniel,1,0,False,1,1,True
15,PA,PA-30,https://github.com/qin-team-recipe/05-recipe-a...,https://chat.openai.com/share/4a9bc421-35fb-44...,Context,Richard,Daniel,1,2,False,-1,1,True
18,PA,PA-12,https://github.com/globalbibletools/gbt/pull/114,https://chat.openai.com/share/1e32601b-3ca7-40...,Context,Richard,Daniel,2,1,False,1,1,True
19,PA,PA-12,https://github.com/globalbibletools/gbt/pull/114,https://chat.openai.com/share/1e32601b-3ca7-40...,Specificity,Richard,Daniel,2,1,False,1,1,True
